In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.cluster import KMeans

data = load_breast_cancer()

df = pd.DataFrame(data.data, columns=data.feature_names)

df["target"] = data.target

df.info()#visualizza le informazioni principali del DataFrame

df.describe()#visualizza le statistiche di base del DataFrame

df.tail(10)#visualizza le ultime 10 righe del DataFrame

corr_matrix = df.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm")
plt.show()

X = df.drop("target", axis=1)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

kmeans.fit(X)

print(kmeans.inertia_)

In [ ]:
#Esercizio 2
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import f1_score

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

tree_clf = DecisionTreeClassifier(max_depth=15, random_state=42)
tree_clf.fit(X_train, y_train)

y_pred_tree = tree_clf.predict(X_test)

f1_tree = f1_score(y_test, y_pred_tree)

print("F1 Decision Tree:", f1_tree)
#un altro modo per calcolare f1_score è : cross_val_score(tree_clf,X_train,y_train,cv=5,scoring="f1")
#scoring="f1" va bene per classificazione binaria come il breast_cancer, per la multiclasse va messo scoring="f1_macro"
#stesso vale per f1_score, di default ha parametro average="binary", per la multiclasse va scritto:
#f1_score(y_test, y_pred_tree,average="macro")


knn_clf = KNeighborsClassifier(n_neighbors=5)
knn_clf.fit(X_train, y_train)

y_pred_knn = knn_clf.predict(X_test)

f1_knn = f1_score(y_test, y_pred_knn)

print("F1 KNN:", f1_knn)


voting_clf = VotingClassifier(
    estimators=[
        ("tree", tree_clf),
        ("knn", knn_clf)
    ],
    voting="hard"
)

voting_clf.fit(X_train, y_train)

y_pred_voting = voting_clf.predict(X_test)

f1_voting = f1_score(y_test, y_pred_voting)

print("F1 Voting Classifier:", f1_voting)

In [ ]:
#Esercizio 3
import torch
import torch.nn as nn
import torch.nn.functional as F

class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.fc1 = nn.Linear(128, 64)
        self.fc2 = nn.Linear(64, 16)
        self.fc3 = nn.Linear(16, 4)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.softmax(self.fc3(x), dim=1)
        return x
    
#Non è corretto usare dim=0, perché l’output ha shape 
# (batch_size, num_classes) e la Softmax deve essere applicata sulle
# classi di ogni esempio, quindi sulla dimensione 1. 
# Con dim=0 normalizzerei lungo il batch, confrontando esempi 
# diversi tra loro.
